In [2]:
%pip install --quiet pandas pyarrow geopandas shapely matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Pipeline ETL — Setor Censitário × CEP × Renda (Brasil completo)

Versão Brasil do `1_pipeline_etl_sp.ipynb`. Mesmo padrão de 8 etapas, mas itera sobre os 27 arquivos CNEFE (um por UF) e roda imputação cascateada respeitando município/distrito/subdistrito/bairro (chaves nunca cruzam UF, pois `cod_mun` já tem prefixo da UF).

**Entradas (todas do IBGE):**
- CNEFE 2022: 27 CSVs em `saida_cnefe_uf/extraido/SEM_UF/<UF_CODIGO>_<UF_SIGLA>.csv`
- Shapefile CD2022: `BR_setores_CD2022/BR_setores_CD2022.shp`
- Agregados de renda IBGE: `Agregados_por_setores_renda_responsavel_BR.xlsx`

**Saída final em `saida_etl_final_brasil/`:**
- `renda_setor_cep_brasil_final.parquet`
- `renda_setor_cep_brasil_final.csv` (opcional — pesado, ~1GB)
- `renda_setor_cep_brasil_final_resumo.json`

**Resumibilidade**: SQLite intermediário com tabela `status_ingest` que registra cada CNEFE processado. Se o kernel cair, religue com `REBUILD_SQLITE=False` para retomar de onde parou.

**Modo smoke**: por default `TEST_MODE=True` roda só 3 UFs pequenas (RO, AC, RR) — valida o pipeline end-to-end em ~30-60 min antes do full run.

## Setup — imports, paths e parâmetros

In [3]:
from pathlib import Path
import json
import sqlite3
import time
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

import geopandas as gpd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')


def log(msg):
    print(msg, flush=True)


def now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [4]:
def find_project_root(start):
    start = Path(start).resolve()
    for c in [start, *start.parents]:
        if (c / 'saida_cnefe_uf').exists() and (c / 'BR_setores_CD2022').exists():
            return c
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CNEFE_ROOT   = PROJECT_ROOT / 'saida_cnefe_uf' / 'extraido' / 'SEM_UF'
RENDA_FILE   = PROJECT_ROOT / 'Agregados_por_setores_renda_responsavel_BR_csv' / 'Agregados_por_setores_renda_responsavel_BR.xlsx'
SHAPEFILE    = PROJECT_ROOT / 'BR_setores_CD2022' / 'BR_setores_CD2022.shp'

OUTPUT_DIR   = PROJECT_ROOT / 'saida_etl_final_brasil'
WORK_SQLITE  = OUTPUT_DIR / 'etl_final_brasil_work.sqlite'
OUT_PARQUET  = OUTPUT_DIR / 'renda_setor_cep_brasil_final.parquet'
OUT_CSV      = OUTPUT_DIR / 'renda_setor_cep_brasil_final.csv'
OUT_RESUMO   = OUTPUT_DIR / 'renda_setor_cep_brasil_final_resumo.json'

# ===========================================================================
# MODO DE EXECUCAO
# ===========================================================================
# TEST_MODE=True  -> roda apenas TEST_UFS (smoke ~30-60 min, valida pipeline)
# TEST_MODE=False -> Brasil completo (~2-6 h, todos os 27 CNEFE)
TEST_MODE = True
TEST_UFS  = ['RO', 'AC', 'RR']   # smoke: 3 UFs pequenas

# Filtro UF arbitrario (None = sem filtro). Sobrescreve TEST_UFS se TEST_MODE=True.
UF_FILTER = TEST_UFS if TEST_MODE else None

# Reconstruir SQLite do zero? False = reaproveita checkpoints anteriores (retomada).
REBUILD_SQLITE = True

# Performance
CHUNKSIZE = 250_000

# Exportar CSV? (parquet sempre. CSV no Brasil completo pode passar de 1GB.)
EXPORT_CSV = True

# Imputacao
TIPOS_IMPUTAVEIS = {'0', '1'}    # CD_TIPO 0=urbano comum, 1=rural comum
MIN_VIZINHOS = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT     :', PROJECT_ROOT)
print('CNEFE_ROOT       :', CNEFE_ROOT)
print('RENDA_FILE       :', RENDA_FILE)
print('SHAPEFILE        :', SHAPEFILE)
print('OUTPUT_DIR       :', OUTPUT_DIR)
print('TEST_MODE        :', TEST_MODE)
print('UF_FILTER        :', UF_FILTER)
print('REBUILD_SQLITE   :', REBUILD_SQLITE)

PROJECT_ROOT     : C:\Users\matpa\Documents\Base CEP_LOG_RENDA
CNEFE_ROOT       : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_cnefe_uf\extraido\SEM_UF
RENDA_FILE       : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\Agregados_por_setores_renda_responsavel_BR_csv\Agregados_por_setores_renda_responsavel_BR.xlsx
SHAPEFILE        : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\BR_setores_CD2022\BR_setores_CD2022.shp
OUTPUT_DIR       : C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_brasil
TEST_MODE        : True
UF_FILTER        : ['RO', 'AC', 'RR']
REBUILD_SQLITE   : True


In [5]:
UF_POR_SIGLA = {
    'RO':'11','AC':'12','AM':'13','RR':'14','PA':'15','AP':'16','TO':'17',
    'MA':'21','PI':'22','CE':'23','RN':'24','PB':'25','PE':'26','AL':'27',
    'SE':'28','BA':'29','MG':'31','ES':'32','RJ':'33','SP':'35','PR':'41',
    'SC':'42','RS':'43','MS':'50','MT':'51','GO':'52','DF':'53',
}
SIGLA_POR_UF = {v: k for k, v in UF_POR_SIGLA.items()}

REQUIRED_CNEFE_COLUMNS = ['CEP', 'COD_SETOR', 'LATITUDE', 'LONGITUDE', 'NV_GEO_COORD']
REQUIRED_RENDA_COLUMNS = ['CD_SETOR', 'V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']


def normalize_uf_filters(values):
    if not values:
        return None
    out = set()
    for v in values:
        t = str(v).strip().upper()
        if t in UF_POR_SIGLA:
            out.add(UF_POR_SIGLA[t])
        elif t in SIGLA_POR_UF:
            out.add(t)
        else:
            raise ValueError(f'UF invalida: {v}')
    return sorted(out)


def infer_uf_code_from_filename(path):
    prefix = path.stem.split('_', 1)[0].strip().upper()
    if prefix in SIGLA_POR_UF:
        return prefix
    if prefix in UF_POR_SIGLA:
        return UF_POR_SIGLA[prefix]
    return None


def collect_cnefe_files(root, uf_codes=None):
    if not root.exists():
        raise FileNotFoundError(f'Pasta CNEFE nao encontrada: {root}')
    files = sorted(p for p in root.rglob('*.csv') if p.is_file())
    if uf_codes:
        files = [p for p in files if infer_uf_code_from_filename(p) in set(uf_codes)]
    if not files:
        raise ValueError(f'Nenhum CSV CNEFE selecionado (uf_codes={uf_codes}).')
    return files


def parse_br_number(series):
    # XLSX do IBGE armazena numeros como float en-US (ponto decimal).
    # pd.to_numeric converte direto e devolve NaN para 'X' (sigilo).
    return pd.to_numeric(series, errors='coerce')


def open_sqlite(p):
    p.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(p)
    conn.execute('PRAGMA journal_mode = WAL')
    conn.execute('PRAGMA synchronous = NORMAL')
    conn.execute('PRAGMA temp_store = MEMORY')
    conn.execute('PRAGMA cache_size = -200000')
    conn.execute('PRAGMA mmap_size = 30000000000')
    return conn


def reset_tables(conn):
    conn.executescript('''
        DROP TABLE IF EXISTS setor_cep_geo;
        DROP TABLE IF EXISTS setor_cep_orig;
        DROP TABLE IF EXISTS status_ingest;
        CREATE TABLE setor_cep_geo (
            cd_setor TEXT NOT NULL,
            cep TEXT NOT NULL,
            qtd_enderecos INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep)
        ) WITHOUT ROWID;
        CREATE TABLE setor_cep_orig (
            cd_setor TEXT NOT NULL,
            cep TEXT NOT NULL,
            qtd_enderecos INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep)
        ) WITHOUT ROWID;
        CREATE TABLE status_ingest (
            cnefe_file    TEXT PRIMARY KEY,
            rows_read     INTEGER NOT NULL,
            via_geo       INTEGER NOT NULL,
            via_orig_only INTEGER NOT NULL,
            finished_at   TEXT NOT NULL
        );
    ''')
    conn.commit()

## Etapa 1 — Renda como driver (XLSX)

Lê o XLSX do IBGE com renda média (V06004) e mediana (V06006). Classifica `motivo_renda` em `numerica` / `sigilo` (X) / `ausente`. Filtra pelo `UF_FILTER` se configurado.

In [6]:
def load_renda(path, uf_codes=None):
    log(f'[renda] Lendo {path}')
    df = pd.read_excel(path, dtype=str, keep_default_na=False, na_filter=False)
    missing = [c for c in REQUIRED_RENDA_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f'Colunas ausentes: {missing}')
    df['cd_setor'] = df['CD_SETOR'].astype(str).str.strip()
    df = df.loc[df['cd_setor'].str.len() == 15].copy()
    if uf_codes:
        df = df.loc[df['cd_setor'].str.slice(0, 2).isin(set(uf_codes))].copy()

    vars_renda = ['V06001','V06002','V06003','V06004','V06005','V06006']
    for v in vars_renda:
        df[f'raw_{v.lower()}'] = df[v].astype(str).str.strip()
        df[f'renda_{v.lower()}'] = parse_br_number(df[v])

    def classify(row):
        if pd.notna(row['renda_v06004']):
            return 'numerica'
        if row['raw_v06004'].upper() == 'X':
            return 'sigilo'
        return 'ausente'

    df['motivo_renda'] = df.apply(classify, axis=1)
    keep = ['cd_setor', 'motivo_renda'] + [f'renda_{v.lower()}' for v in vars_renda]
    out = df[keep].drop_duplicates('cd_setor', keep='last').reset_index(drop=True)
    log(f'[renda] Setores no driver: {len(out):,}')
    print(out['motivo_renda'].value_counts())
    return out


uf_codes = normalize_uf_filters(UF_FILTER)
renda_df = load_renda(RENDA_FILE, uf_codes=uf_codes)
display(renda_df.head())

[renda] Lendo C:\Users\matpa\Documents\Base CEP_LOG_RENDA\Agregados_por_setores_renda_responsavel_BR_csv\Agregados_por_setores_renda_responsavel_BR.xlsx
[renda] Setores no driver: 7,043
motivo_renda
numerica    6289
sigilo       532
ausente      222
Name: count, dtype: int64


,cd_setor,motivo_renda,renda_v06001,renda_v06002,renda_v06003,renda_v06004,renda_v06005,renda_v06006
0,110001505000002,numerica,336.0000,928.0000,1.6700,"2,453.0300","13,047,638.9300","1,212.0000"
1,110001505000003,numerica,208.0000,556.0000,1.7900,"2,126.4400","2,348,222.3900","1,500.0000"
2,110001505000004,numerica,85.0000,222.0000,1.6000,"1,664.9400","1,270,082.0600","1,200.0000"
3,110001505000006,numerica,281.0000,783.0000,2.4100,"1,722.4000","1,446,560.6000","1,200.0000"
4,110001505000007,numerica,291.0000,748.0000,1.3800,"1,930.9400","2,255,368.8700","1,212.0000"


## Etapa 2 — Geofencing (loop de UFs do CNEFE)

Para cada CSV CNEFE selecionado:
1. Constrói GeoDataFrame de pontos (LATITUDE, LONGITUDE).
2. `gpd.sjoin(..., predicate='within')` contra `shp_geo` → atribui `cd_setor` pela geometria.
3. Grava em DUAS tabelas SQLite:
   - `setor_cep_geo`: cd_setor da geometria (primário)
   - `setor_cep_orig`: COD_SETOR original do CNEFE (fallback)
4. Registra checkpoint em `status_ingest` ao final de cada arquivo.

Se kernel cair: reinicie com `REBUILD_SQLITE=False` para pular CNEFE já processados.

In [7]:
def load_shapefile_geometry(shapefile, uf_codes=None):
    log(f'[shapefile] Carregando geometria de {shapefile}')
    gdf = gpd.read_file(shapefile, columns=['CD_SETOR', 'CD_UF', 'geometry'])
    gdf['cd_setor'] = gdf['CD_SETOR'].astype(str).str.strip()
    gdf['cod_uf']   = gdf['CD_UF'].astype(str).str.strip().str.zfill(2)
    log(f'[shapefile] Total Brasil: {len(gdf):,}  |  CRS: {gdf.crs}')
    if uf_codes:
        gdf = gdf.loc[gdf['cod_uf'].isin(set(uf_codes))]
    gdf = gdf.loc[gdf['cd_setor'].str.len() == 15, ['cd_setor', 'geometry']].reset_index(drop=True)
    log(f'[shapefile] Setores apos filtro UF={uf_codes}: {len(gdf):,}')
    if len(gdf) == 0:
        raise RuntimeError('Filtro UF zerou setores no shapefile.')
    return gdf


shp_geo = load_shapefile_geometry(SHAPEFILE, uf_codes=uf_codes)
display(shp_geo.head(3))

[shapefile] Carregando geometria de C:\Users\matpa\Documents\Base CEP_LOG_RENDA\BR_setores_CD2022\BR_setores_CD2022.shp
[shapefile] Total Brasil: 468,099  |  CRS: EPSG:4674
[shapefile] Setores apos filtro UF=['11', '12', '14']: 7,454


,cd_setor,geometry
0,120040105000708,"POLYGON ((-67.83814 -9.93462, -67.83862 -9.935..."
1,110001505000002,"POLYGON ((-61.99996 -11.94202, -62.0057 -11.94..."
2,110001505000003,"POLYGON ((-62.00377 -11.9294, -62.00375 -11.92..."


In [8]:
def ingest_cnefe(conn, files, shp_geo, chunksize, skip_files=None):
    target_crs = shp_geo.crs
    skip_files = set(skip_files or [])
    stats = {'files_processed': 0, 'rows_read': 0, 'via_geo': 0, 'via_orig_only': 0, 'invalidos': 0}

    for i, csv_path in enumerate(files, 1):
        if csv_path.name in skip_files:
            log(f'[skip] {csv_path.name} (ja processado)')
            continue
        log(f'[cnefe] {i}/{len(files)}: {csv_path.name}  ({csv_path.stat().st_size/1e9:.2f} GB)')
        t_file = time.time()
        file_rows = 0
        file_via_geo = 0
        file_via_orig = 0
        reader = pd.read_csv(
            csv_path, sep=';', dtype=str,
            usecols=REQUIRED_CNEFE_COLUMNS,
            keep_default_na=False, na_filter=False, chunksize=chunksize,
        )
        chunk_i = 0
        for chunk in reader:
            chunk_i += 1
            raw = len(chunk)
            if raw == 0:
                continue
            lat = pd.to_numeric(chunk['LATITUDE'].str.replace(',', '.', regex=False), errors='coerce')
            lng = pd.to_numeric(chunk['LONGITUDE'].str.replace(',', '.', regex=False), errors='coerce')
            cep_digits = chunk['CEP'].fillna('').astype(str).str.replace(r'\D+', '', regex=True)
            cep = cep_digits.where(cep_digits.eq(''), cep_digits.str.zfill(8))
            cnefe_setor = chunk['COD_SETOR'].fillna('').astype(str).str.strip().str.slice(0, 15)
            mask = lat.notna() & lng.notna() & cep.ne('') & cnefe_setor.str.len().eq(15)
            invalid = int((~mask).sum())
            if not mask.any():
                stats['rows_read'] += raw
                stats['invalidos'] += invalid
                file_rows += raw
                continue

            # setor_cep_orig (sempre).
            orig_df = pd.DataFrame({
                'cd_setor': cnefe_setor[mask].values,
                'cep': cep[mask].values,
            })
            orig_grouped = (
                orig_df.groupby(['cd_setor', 'cep'], sort=False)
                       .size().reset_index(name='qtd_enderecos')
            )
            with conn:
                conn.executemany(
                    'INSERT INTO setor_cep_orig (cd_setor, cep, qtd_enderecos) VALUES (?, ?, ?) '
                    'ON CONFLICT(cd_setor, cep) DO UPDATE SET '
                    'qtd_enderecos = setor_cep_orig.qtd_enderecos + excluded.qtd_enderecos',
                    orig_grouped.itertuples(index=False, name=None),
                )

            # Geofencing -> setor_cep_geo.
            pts = gpd.GeoDataFrame(
                {'cep': cep[mask].values},
                geometry=gpd.points_from_xy(lng[mask], lat[mask]),
                crs=target_crs,
            )
            joined = gpd.sjoin(pts, shp_geo, how='left', predicate='within')
            has_geo = joined['cd_setor'].notna()
            if has_geo.any():
                geo_df = pd.DataFrame({
                    'cd_setor': joined.loc[has_geo, 'cd_setor'].values,
                    'cep': joined.loc[has_geo, 'cep'].values,
                })
                geo_grouped = (
                    geo_df.groupby(['cd_setor', 'cep'], sort=False)
                          .size().reset_index(name='qtd_enderecos')
                )
                with conn:
                    conn.executemany(
                        'INSERT INTO setor_cep_geo (cd_setor, cep, qtd_enderecos) VALUES (?, ?, ?) '
                        'ON CONFLICT(cd_setor, cep) DO UPDATE SET '
                        'qtd_enderecos = setor_cep_geo.qtd_enderecos + excluded.qtd_enderecos',
                        geo_grouped.itertuples(index=False, name=None),
                    )
            via_g = int(has_geo.sum())
            via_o = int((~has_geo).sum())
            stats['rows_read'] += raw
            stats['via_geo'] += via_g
            stats['via_orig_only'] += via_o
            stats['invalidos'] += invalid
            file_rows += raw
            file_via_geo += via_g
            file_via_orig += via_o
            if chunk_i % 4 == 1:
                log(f'  chunk {chunk_i}: {raw:,} | via_geo={via_g:,} | so_orig={via_o:,}')

        stats['files_processed'] += 1
        # Checkpoint do arquivo.
        with conn:
            conn.execute(
                'INSERT OR REPLACE INTO status_ingest '
                '(cnefe_file, rows_read, via_geo, via_orig_only, finished_at) VALUES (?, ?, ?, ?, ?)',
                (csv_path.name, file_rows, file_via_geo, file_via_orig, now_iso()),
            )
        dt = time.time() - t_file
        log(f'[done] {csv_path.name} em {dt/60:.1f} min | {file_rows:,} linhas')
    return stats


cnefe_files = collect_cnefe_files(CNEFE_ROOT, uf_codes=uf_codes)
log(f'Arquivos CNEFE selecionados: {len(cnefe_files)}')
for p in cnefe_files[:10]:
    log(f'  {p.name}  ({p.stat().st_size/1e9:.2f} GB)')
if len(cnefe_files) > 10:
    log(f'  ... +{len(cnefe_files)-10} arquivos')

t0 = time.time()
with open_sqlite(WORK_SQLITE) as conn:
    if REBUILD_SQLITE:
        reset_tables(conn)
        skip_files = set()
    else:
        skip_files = {row[0] for row in conn.execute('SELECT cnefe_file FROM status_ingest').fetchall()}
        log(f'Reaproveitando SQLite. Pulando {len(skip_files)} arquivos ja processados.')

    cnefe_stats = ingest_cnefe(conn, cnefe_files, shp_geo, CHUNKSIZE, skip_files=skip_files)
    conn.execute('CREATE INDEX IF NOT EXISTS idx_geo_cd  ON setor_cep_geo(cd_setor)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_orig_cd ON setor_cep_orig(cd_setor)')
    conn.commit()
    total_geo  = conn.execute('SELECT COUNT(*) FROM setor_cep_geo').fetchone()[0]
    total_orig = conn.execute('SELECT COUNT(*) FROM setor_cep_orig').fetchone()[0]

log(f'\n[etapa 2] Tempo: {(time.time()-t0)/60:.1f} min')
log(f'[etapa 2] Pares (setor, cep) em setor_cep_geo : {total_geo:,}')
log(f'[etapa 2] Pares (setor, cep) em setor_cep_orig: {total_orig:,}')
log(f'[etapa 2] Stats: {cnefe_stats}')

Arquivos CNEFE selecionados: 3
  11_RO.csv  (0.15 GB)
  12_AC.csv  (0.07 GB)
  14_RR.csv  (0.04 GB)
[cnefe] 1/3: 11_RO.csv  (0.15 GB)
  chunk 1: 250,000 | via_geo=250,000 | so_orig=0
[done] 11_RO.csv em 0.1 min | 965,370 linhas
[cnefe] 2/3: 12_AC.csv  (0.07 GB)
  chunk 1: 250,000 | via_geo=249,997 | so_orig=3
[done] 12_AC.csv em 0.0 min | 410,524 linhas
[cnefe] 3/3: 14_RR.csv  (0.04 GB)
  chunk 1: 250,000 | via_geo=250,000 | so_orig=0
[done] 14_RR.csv em 0.0 min | 260,515 linhas

[etapa 2] Tempo: 0.1 min
[etapa 2] Pares (setor, cep) em setor_cep_geo : 33,797
[etapa 2] Pares (setor, cep) em setor_cep_orig: 29,917
[etapa 2] Stats: {'files_processed': 3, 'rows_read': 1636409, 'via_geo': 1636384, 'via_orig_only': 25, 'invalidos': 0}


## Etapa 3 — Agregar setor → CEPs (cascata híbrida)

Agrupa CEPs por setor a partir do SQLite. Setores presentes em `setor_cep_geo` ficam com `origem_cep = geofencing`. Setores ausentes em `geo` mas presentes em `orig` viram fallback `cnefe_original`.

In [9]:
AGG_SQL = '''
    SELECT cd_setor,
           COUNT(*) AS qtd_ceps,
           MIN(cep) AS cep_inicial,
           MAX(cep) AS cep_final,
           CASE WHEN MIN(cep) = MAX(cep) THEN MIN(cep)
                ELSE MIN(cep) || ' - ' || MAX(cep) END AS faixa_cep,
           SUM(qtd_enderecos) AS total_enderecos,
           GROUP_CONCAT(cep, '|') AS lista_ceps
    FROM {tabela}
    GROUP BY cd_setor
'''

with open_sqlite(WORK_SQLITE) as conn:
    agg_geo  = pd.read_sql_query(AGG_SQL.format(tabela='setor_cep_geo'),  conn)
    agg_orig = pd.read_sql_query(AGG_SQL.format(tabela='setor_cep_orig'), conn)

log(f'agg_geo  : {len(agg_geo):,}')
log(f'agg_orig : {len(agg_orig):,}')

if uf_codes:
    agg_geo  = agg_geo.loc[agg_geo['cd_setor'].str.slice(0, 2).isin(set(uf_codes))].copy()
    agg_orig = agg_orig.loc[agg_orig['cd_setor'].str.slice(0, 2).isin(set(uf_codes))].copy()

setores_em_geo = set(agg_geo['cd_setor'])
agg_orig_only  = agg_orig.loc[~agg_orig['cd_setor'].isin(setores_em_geo)].copy()

agg_geo['origem_cep']       = 'geofencing'
agg_orig_only['origem_cep'] = 'cnefe_original'

setor_ceps = pd.concat([agg_geo, agg_orig_only], ignore_index=True)
log(f'Setores agregados (geo + fallback orig): {len(setor_ceps):,}')
print(setor_ceps['origem_cep'].value_counts())
display(setor_ceps.head())

agg_geo  : 7,131
agg_orig : 6,290
Setores agregados (geo + fallback orig): 7,912
origem_cep
geofencing        7131
cnefe_original     781
Name: count, dtype: int64


,cd_setor,qtd_ceps,cep_inicial,cep_final,faixa_cep,total_enderecos,lista_ceps,origem_cep
0,110001505000002,1,76954000,76954000,76954000,404,76954000,geofencing
1,110001505000003,1,76954000,76954000,76954000,264,76954000,geofencing
2,110001505000004,1,76954000,76954000,76954000,122,76954000,geofencing
3,110001505000006,1,76954000,76954000,76954000,383,76954000,geofencing
4,110001505000007,1,76954000,76954000,76954000,387,76954000,geofencing


## Etapa 4 — Atributos do shapefile

Carrega atributos sem geometria, **incluindo CD_BAIRRO, CD_DIST, CD_SUBDIST** para a imputação cascateada na Etapa 7.

In [10]:
shp_attrs = gpd.read_file(
    SHAPEFILE, ignore_geometry=True,
    columns=['CD_SETOR','CD_UF','NM_UF','CD_MUN','NM_MUN','CD_DIST','NM_DIST',
             'CD_SUBDIST','NM_SUBDIST','CD_BAIRRO','NM_BAIRRO','SITUACAO','CD_TIPO','AREA_KM2'],
)
shp_attrs['cd_setor'] = shp_attrs['CD_SETOR'].astype(str).str.strip()
shp_attrs['cod_uf']   = shp_attrs['CD_UF'].astype(str).str.strip().str.zfill(2)
shp_attrs['cod_mun']  = shp_attrs['CD_MUN'].astype(str).str.strip().str.zfill(7)
shp_attrs['nm_uf']    = shp_attrs['NM_UF'].astype(str).str.strip()
shp_attrs['nm_mun']   = shp_attrs['NM_MUN'].astype(str).str.strip()
shp_attrs['area_km2'] = pd.to_numeric(shp_attrs['AREA_KM2'], errors='coerce')
shp_attrs = shp_attrs.loc[shp_attrs['cd_setor'].str.len() == 15]
if uf_codes:
    shp_attrs = shp_attrs.loc[shp_attrs['cod_uf'].isin(set(uf_codes))]
shp_attrs = shp_attrs.drop_duplicates('cd_setor').reset_index(drop=True)
log(f'Setores no shapefile: {len(shp_attrs):,}')
display(shp_attrs.head(2))

Setores no shapefile: 7,454


,CD_SETOR,SITUACAO,CD_TIPO,AREA_KM2,CD_UF,NM_UF,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_SUBDIST,NM_SUBDIST,CD_BAIRRO,NM_BAIRRO,cd_setor,cod_uf,cod_mun,nm_uf,nm_mun,area_km2
0,120040105000708,Urbana,0,0.0385,12,Acre,1200401,Rio Branco,120040105,Rio Branco,12004010500,NaN,NaN,NaN,120040105000708,12,1200401,Acre,Rio Branco,0.0385
1,110001505000002,Urbana,0,0.5393,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,NaN,1100015006,Redondo,110001505000002,11,1100015,Rondônia,Alta Floresta D'Oeste,0.5393


## Etapa 5 — Outer join (driver=renda) e categoria `fora_csv_ibge`

Junta renda (driver) com `setor_ceps` (outer join). Setores que estão na malha CD2022 com CEP via geofencing mas o IBGE não publicou renda recebem `motivo_renda = fora_csv_ibge`.

In [11]:
out = renda_df.merge(setor_ceps, on='cd_setor', how='outer')
veio_so_de_cnefe = out['motivo_renda'].isna() & out['qtd_ceps'].notna()
out.loc[veio_so_de_cnefe, 'motivo_renda'] = 'fora_csv_ibge'

out['tem_cep'] = out['qtd_ceps'].notna().astype(int)
for col in ['qtd_ceps', 'total_enderecos']:
    out[col] = out[col].fillna(0).astype('int64')

out = out.merge(shp_attrs, on='cd_setor', how='left')
out['sigla_uf'] = out['cd_setor'].str.slice(0, 2).map(SIGLA_POR_UF)
out['esta_no_shapefile'] = out['cod_uf'].notna().astype(int)

# Mantem apenas setores que existem no shapefile CD2022.
n_pre = len(out)
out = out.loc[out['esta_no_shapefile'] == 1].copy()
log(f'Filtro shapefile: {n_pre:,} -> {len(out):,} (removidos {n_pre - len(out):,} codigos antigos do CNEFE)')

log(f'  com renda no CSV (numerica/sigilo/ausente): {int(out["motivo_renda"].isin(["numerica","sigilo","ausente"]).sum()):,}')
log(f'  fora_csv_ibge (sem renda mas com CEP): {int((out["motivo_renda"] == "fora_csv_ibge").sum()):,}')
display(out.head(3))

Filtro shapefile: 7,919 -> 7,172 (removidos 747 codigos antigos do CNEFE)
  com renda no CSV (numerica/sigilo/ausente): 7,043
  fora_csv_ibge (sem renda mas com CEP): 129


,cd_setor,motivo_renda,renda_v06001,renda_v06002,renda_v06003,renda_v06004,renda_v06005,renda_v06006,qtd_ceps,cep_inicial,cep_final,faixa_cep,total_enderecos,lista_ceps,origem_cep,tem_cep,CD_SETOR,SITUACAO,CD_TIPO,AREA_KM2,CD_UF,NM_UF,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_SUBDIST,NM_SUBDIST,CD_BAIRRO,NM_BAIRRO,cod_uf,cod_mun,nm_uf,nm_mun,area_km2,sigla_uf,esta_no_shapefile
1,110001505000002,numerica,336.0000,928.0000,1.6700,"2,453.0300","13,047,638.9300","1,212.0000",1,76954000,76954000,76954000,404,76954000,geofencing,1,110001505000002,Urbana,0,0.5393,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,NaN,1100015006,Redondo,11,1100015,Rondônia,Alta Floresta D'Oeste,0.5393,RO,1
2,110001505000003,numerica,208.0000,556.0000,1.7900,"2,126.4400","2,348,222.3900","1,500.0000",1,76954000,76954000,76954000,264,76954000,geofencing,1,110001505000003,Urbana,0,0.2362,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,NaN,1100015006,Redondo,11,1100015,Rondônia,Alta Floresta D'Oeste,0.2362,RO,1
3,110001505000004,numerica,85.0000,222.0000,1.6000,"1,664.9400","1,270,082.0600","1,200.0000",1,76954000,76954000,76954000,122,76954000,geofencing,1,110001505000004,Urbana,0,0.2119,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,NaN,1100015005,Princesa Isabel,11,1100015,Rondônia,Alta Floresta D'Oeste,0.2119,RO,1


## Etapa 6 — Estatísticas intermediárias (antes da imputação)

In [12]:
stats_inter = pd.DataFrame({
    'metrica': [
        'setores no output total',
        'com CEP (tem_cep=1)',
        'sem CEP (tem_cep=0)',
        'renda numerica',
        'renda sigilo (X)',
        'renda ausente',
        'fora_csv_ibge (com CEP, sem renda)',
        'origem_cep geofencing',
        'origem_cep cnefe_original (fallback)',
    ],
    'valor': [
        len(out),
        int(out['tem_cep'].sum()),
        int((out['tem_cep'] == 0).sum()),
        int((out['motivo_renda'] == 'numerica').sum()),
        int((out['motivo_renda'] == 'sigilo').sum()),
        int((out['motivo_renda'] == 'ausente').sum()),
        int((out['motivo_renda'] == 'fora_csv_ibge').sum()),
        int((out['origem_cep'] == 'geofencing').sum()),
        int((out['origem_cep'] == 'cnefe_original').sum()),
    ],
})
display(stats_inter)

# Cobertura por UF (so faz sentido se rodando varias UFs)
uf_stats = (
    out.groupby('sigla_uf')
       .agg(setores=('cd_setor', 'count'),
            com_cep=('tem_cep', 'sum'),
            via_geo=('origem_cep', lambda s: (s == 'geofencing').sum()),
            via_orig=('origem_cep', lambda s: (s == 'cnefe_original').sum()),
            renda_num=('motivo_renda', lambda s: (s == 'numerica').sum()))
       .assign(pct_cep=lambda d: (d['com_cep'] / d['setores'] * 100).round(2),
               pct_renda_num=lambda d: (d['renda_num'] / d['setores'] * 100).round(2))
       .sort_values('setores', ascending=False)
)
print('Cobertura por UF (pre-imputacao):')
display(uf_stats)

,metrica,valor
0,setores no output total,7172
1,com CEP (tem_cep=1),7165
2,sem CEP (tem_cep=0),7
3,renda numerica,6289
4,renda sigilo (X),532
5,renda ausente,222
6,"fora_csv_ibge (com CEP, sem renda)",129
7,origem_cep geofencing,7131
8,origem_cep cnefe_original (fallback),34


Cobertura por UF (pre-imputacao):


,setores,com_cep,via_geo,via_orig,renda_num,pct_cep,pct_renda_num
sigla_uf,,,,,,,
RO,3349,3344,3339,5,3135,99.8500,93.6100
AC,2144,2144,2140,4,1981,100.0000,92.4000
RR,1679,1677,1652,25,1173,99.8800,69.8600


## Etapa 7 — Imputação de renda em cascata

Para setores sem renda numérica com `CD_TIPO ∈ {0, 1}` (urbano comum e rural comum), usa a mediana dos vizinhos no nível mais granular com `n_amostra ≥ MIN_VIZINHOS`:

1. bairro (`cod_mun + CD_DIST + CD_SUBDIST + CD_BAIRRO`)
2. subdistrito
3. distrito
4. município (fallback)

`cod_mun` já tem prefixo da UF (`35` para SP, `12` para AC, etc.), então a cascata **nunca cruza UFs** mesmo rodando Brasil inteiro.

Setores com `CD_TIPO ∈ {2..9}` (especiais — parques, militar, indígenas, favelas com sigilo) ficam `sem_dado_legitimo_tipo_X` — não imputa por construção.

In [13]:
NIVEIS = [
    ('bairro',      ['cod_mun', 'CD_DIST', 'CD_SUBDIST', 'CD_BAIRRO']),
    ('subdistrito', ['cod_mun', 'CD_DIST', 'CD_SUBDIST']),
    ('distrito',    ['cod_mun', 'CD_DIST']),
    ('municipio',   ['cod_mun']),
]


def build_lookups(com_renda_df, niveis):
    out_ = {}
    for nome, keys in niveis:
        agg = (com_renda_df.groupby(keys, dropna=False)
                          .agg(median_v06004=('renda_v06004', 'median'),
                               median_v06006=('renda_v06006', 'median'),
                               n_amostra=('renda_v06004', 'count'))
                          .reset_index())
        out_[nome] = (keys, agg)
    return out_


com_renda = out.loc[out['renda_v06004'].notna()].copy()
log(f'Amostra para lookup (setores com renda numerica): {len(com_renda):,}')
lookups = build_lookups(com_renda, NIVEIS)
for nome, (keys, agg) in lookups.items():
    com_amostra = int((agg['n_amostra'] >= MIN_VIZINHOS).sum())
    log(f'  {nome:11} | grupos: {len(agg):,}  |  com >= {MIN_VIZINHOS} amostras: {com_amostra:,}')

Amostra para lookup (setores com renda numerica): 6,289
  bairro      | grupos: 644  |  com >= 5 amostras: 246
  subdistrito | grupos: 151  |  com >= 5 amostras: 128
  distrito    | grupos: 146  |  com >= 5 amostras: 123
  municipio   | grupos: 89  |  com >= 5 amostras: 89


In [14]:
def apply_imputation(df, lookups, tipos_imp, min_n):
    res = df.copy()
    res['renda_v06004_estimada'] = res['renda_v06004']
    res['renda_v06006_estimada'] = res['renda_v06006']
    res['origem_renda'] = 'original'

    sem_renda_mask = res['renda_v06004'].isna()
    res.loc[sem_renda_mask, 'origem_renda'] = pd.NA

    tipo_str = res['CD_TIPO'].astype(str)
    nao_imp_mask = sem_renda_mask & ~tipo_str.isin(tipos_imp)
    res.loc[nao_imp_mask, 'origem_renda'] = 'sem_dado_legitimo_tipo_' + tipo_str[nao_imp_mask]

    needs_imp = sem_renda_mask & tipo_str.isin(tipos_imp)
    log(f'Candidatos a imputacao (sem renda + CD_TIPO in {sorted(tipos_imp)}): {int(needs_imp.sum()):,}')

    for nome, (keys, agg) in lookups.items():
        if not needs_imp.any():
            break
        agg_valid = agg.loc[agg['n_amostra'] >= min_n, keys + ['median_v06004', 'median_v06006']]
        if agg_valid.empty:
            continue
        candidatos = res.loc[needs_imp].reset_index().rename(columns={'index': '_orig_idx'})
        merged = candidatos.merge(agg_valid, on=keys, how='left')
        ok = merged['median_v06004'].notna()
        if ok.any():
            orig_idx = merged.loc[ok, '_orig_idx'].values
            res.loc[orig_idx, 'renda_v06004_estimada'] = merged.loc[ok, 'median_v06004'].values
            res.loc[orig_idx, 'renda_v06006_estimada'] = merged.loc[ok, 'median_v06006'].values
            res.loc[orig_idx, 'origem_renda'] = f'imputado_{nome}'
            needs_imp.loc[orig_idx] = False
            log(f'  nivel {nome:11}: {int(ok.sum()):,} setores imputados')

    res.loc[needs_imp, 'origem_renda'] = 'imputacao_sem_amostra'
    return res


out_final = apply_imputation(out, lookups, TIPOS_IMPUTAVEIS, MIN_VIZINHOS)
log(f'Linhas no output final: {len(out_final):,}')

Candidatos a imputacao (sem renda + CD_TIPO in ['0', '1']): 185
  nivel bairro     : 181 setores imputados
  nivel subdistrito: 1 setores imputados
  nivel municipio  : 3 setores imputados
Linhas no output final: 7,172


## Etapa 8 — Estatísticas finais e exportar

In [15]:
print('Distribuicao de origem_renda:')
origem_counts = out_final['origem_renda'].value_counts(dropna=False)
origem_pct = (origem_counts / len(out_final) * 100).round(2)
display(pd.DataFrame({'setores': origem_counts, 'pct': origem_pct}))

n_total       = len(out_final)
n_original    = int((out_final['origem_renda'] == 'original').sum())
n_imputados   = int(out_final['origem_renda'].astype(str).str.startswith('imputado_').sum())
n_legitimo    = int(out_final['origem_renda'].astype(str).str.startswith('sem_dado_legitimo_').sum())
n_renda_final = int(out_final['renda_v06004_estimada'].notna().sum())
n_com_cep     = int((out_final['tem_cep'] == 1).sum())

print()
print(f'Total setores              : {n_total:,}')
print(f'  com renda original       : {n_original:,} ({n_original/n_total*100:.2f}%)')
print(f'  com renda imputada       : {n_imputados:,} ({n_imputados/n_total*100:.2f}%)')
print(f'  legitimos sem renda      : {n_legitimo:,} ({n_legitimo/n_total*100:.2f}%)')
print()
print(f'COBERTURA RENDA (origin+imp): {n_renda_final:,} / {n_total:,} = {n_renda_final/n_total*100:.2f}%')
print(f'COBERTURA CEP              : {n_com_cep:,} / {n_total:,} = {n_com_cep/n_total*100:.2f}%')

# Cobertura final por UF
uf_final = (
    out_final.groupby('sigla_uf')
       .agg(setores=('cd_setor', 'count'),
            com_cep=('tem_cep', 'sum'),
            com_renda=('renda_v06004_estimada', lambda s: s.notna().sum()))
       .assign(pct_cep=lambda d: (d['com_cep'] / d['setores'] * 100).round(2),
               pct_renda=lambda d: (d['com_renda'] / d['setores'] * 100).round(2))
       .sort_values('setores', ascending=False)
)
print()
print('Cobertura final por UF:')
display(uf_final)

Distribuicao de origem_renda:


,setores,pct
origem_renda,,
original,6289,87.6900
sem_dado_legitimo_tipo_5,567,7.9100
imputado_bairro,181,2.5200
sem_dado_legitimo_tipo_4,65,0.9100
sem_dado_legitimo_tipo_6,47,0.6600
sem_dado_legitimo_tipo_3,11,0.1500
sem_dado_legitimo_tipo_2,5,0.0700
imputado_municipio,3,0.0400
sem_dado_legitimo_tipo_7,2,0.0300



Total setores              : 7,172
  com renda original       : 6,289 (87.69%)
  com renda imputada       : 185 (2.58%)
  legitimos sem renda      : 698 (9.73%)

COBERTURA RENDA (origin+imp): 6,474 / 7,172 = 90.27%
COBERTURA CEP              : 7,165 / 7,172 = 99.90%

Cobertura final por UF:


,setores,com_cep,com_renda,pct_cep,pct_renda
sigla_uf,,,,,
RO,3349,3344,3209,99.8500,95.8200
AC,2144,2144,2031,100.0000,94.7300
RR,1679,1677,1234,99.8800,73.5000


In [16]:
# Ordem final das colunas (mesmo padrao do 1_pipeline_etl_sp.ipynb).
final_columns = [
    'cd_setor', 'sigla_uf', 'cod_uf', 'nm_uf',
    'cod_mun', 'nm_mun', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST',
    'CD_BAIRRO', 'NM_BAIRRO', 'SITUACAO', 'CD_TIPO', 'area_km2',
    'motivo_renda', 'origem_renda',
    'renda_v06001', 'renda_v06002', 'renda_v06003',
    'renda_v06004', 'renda_v06005', 'renda_v06006',
    'renda_v06004_estimada', 'renda_v06006_estimada',
    'origem_cep', 'tem_cep', 'qtd_ceps',
    'cep_inicial', 'cep_final', 'faixa_cep',
    'total_enderecos', 'lista_ceps', 'esta_no_shapefile',
]
ordem = [c for c in final_columns if c in out_final.columns]
out_ordenado = out_final[ordem].copy()

out_ordenado.to_parquet(OUT_PARQUET, index=False)
log(f'[export] Parquet: {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size/1e6:.1f} MB)')

if EXPORT_CSV:
    out_ordenado.to_csv(OUT_CSV, sep=';', index=False, encoding='utf-8')
    log(f'[export] CSV: {OUT_CSV}  ({OUT_CSV.stat().st_size/1e6:.1f} MB)')

resumo = {
    'generated_at': now_iso(),
    'test_mode': TEST_MODE,
    'uf_filter': UF_FILTER,
    'tipos_imputaveis': sorted(TIPOS_IMPUTAVEIS),
    'min_vizinhos': MIN_VIZINHOS,
    'setores_total': n_total,
    'setores_com_renda_original': n_original,
    'setores_imputados': n_imputados,
    'setores_legitimos_sem_renda': n_legitimo,
    'cobertura_renda_pct': round(n_renda_final / n_total * 100, 4),
    'cobertura_cep_pct':   round(n_com_cep / n_total * 100, 4),
    'distribuicao_origem_renda': origem_counts.to_dict(),
    'cobertura_por_uf': uf_final.reset_index().to_dict(orient='records'),
    'cnefe_stats': cnefe_stats,
    'outputs': {
        'parquet': str(OUT_PARQUET),
        'csv':     str(OUT_CSV) if EXPORT_CSV else None,
        'sqlite':  str(WORK_SQLITE),
        'summary': str(OUT_RESUMO),
    },
}
OUT_RESUMO.write_text(json.dumps(resumo, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
log(f'[export] Resumo: {OUT_RESUMO}')
print()
print(json.dumps({k: v for k, v in resumo.items() if k != 'cobertura_por_uf'}, ensure_ascii=False, indent=2, default=str)[:1500])

[export] Parquet: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_brasil\renda_setor_cep_brasil_final.parquet  (0.5 MB)
[export] CSV: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_brasil\renda_setor_cep_brasil_final.csv  (2.1 MB)
[export] Resumo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_brasil\renda_setor_cep_brasil_final_resumo.json

{
  "generated_at": "2026-06-01T00:31:31+00:00",
  "test_mode": true,
  "uf_filter": [
    "RO",
    "AC",
    "RR"
  ],
  "tipos_imputaveis": [
    "0",
    "1"
  ],
  "min_vizinhos": 5,
  "setores_total": 7172,
  "setores_com_renda_original": 6289,
  "setores_imputados": 185,
  "setores_legitimos_sem_renda": 698,
  "cobertura_renda_pct": 90.2677,
  "cobertura_cep_pct": 99.9024,
  "distribuicao_origem_renda": {
    "original": 6289,
    "sem_dado_legitimo_tipo_5": 567,
    "imputado_bairro": 181,
    "sem_dado_legitimo_tipo_4": 65,
    "sem_dado_legitimo_tipo_6": 47,
    "sem_dado_legitimo_tipo_3": 11,
    "se

## Notas operacionais

- **Smoke test**: rodar com `TEST_MODE=True` (default) processa só RO+AC+RR — ~30-60 min. Se os números fizerem sentido, mude para `TEST_MODE=False` e Run All de novo.
- **Retomada**: se kernel cair no meio, abra de novo, mude `REBUILD_SQLITE=False` na célula `paths-config` e dê Run All. O loop CNEFE pula arquivos já processados (via tabela `status_ingest`).
- **Output principal**: `saida_etl_final_brasil/renda_setor_cep_brasil_final.parquet` (1 linha = 1 setor da CD2022 Brasil).
- **Cobertura CEP esperada**: ~99% por UF (mesmo padrão que SP).
- **Cobertura renda esperada**: ~97% por UF (original + cascata).
- **Imputação não cruza UF**: chaves usam `cod_mun` que já tem prefixo da UF.
- **CSV é opcional** (`EXPORT_CSV=False` desliga). No Brasil completo o CSV passa de 1 GB; parquet é ~50-100 MB.
- **Próximos passos**: re-rodar `2_analises_diagnostico.ipynb` e `3_visualizacao_mapa.ipynb` apontando para o parquet do Brasil (ou criar versões `_brasil` desses notebooks).